# Exploratory Data Analysis for Fraud Detection

This notebook performs exploratory data analysis on the transaction data to identify patterns and insights for fraud detection.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from datetime import datetime

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.2)

# Configure plot size
plt.rcParams['figure.figsize'] = (12, 8)

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Add the project root to the path so we can import from src
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
from src import config

## Load the Data

In [ ]:
# Load training data
train_data = pd.read_csv(config.TRAIN_DATA_PATH)

# Load test data
test_data = pd.read_csv(config.TEST_DATA_PATH)

print(f'Training data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

In [ ]:
# Display the first few rows of the training data
train_data.head()

## Data Overview

In [ ]:
# Get information about the data
train_data.info()

In [ ]:
# Get summary statistics
train_data.describe()

In [ ]:
# Check for missing values
missing_values = train_data.isnull().sum()
missing_percentage = (missing_values / len(train_data)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

## Target Variable Analysis

In [ ]:
# Check the distribution of the target variable
fraud_counts = train_data['is_fraud'].value_counts()
fraud_percentage = fraud_counts / len(train_data) * 100

print(f'Fraud distribution:
{fraud_counts}')
print(f'
Fraud percentage:
{fraud_percentage}')

In [ ]:
# Visualize the target variable distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='is_fraud', data=train_data)
plt.title('Distribution of Fraud vs. Non-Fraud Transactions')
plt.xlabel('Is Fraud (1 = Yes, 0 = No)')
plt.ylabel('Count')

# Add count labels
for i, count in enumerate(fraud_counts.values):
    plt.text(i, count + 500, f'{count:,}
({fraud_percentage[i]:.2f}%)', 
             ha='center', va='bottom', fontsize=12)

plt.show()

## Transaction Amount Analysis

In [ ]:
# Analyze transaction amounts
plt.figure(figsize=(12, 6))
sns.histplot(data=train_data, x='amt', hue='is_fraud', bins=50, kde=True, element='step')
plt.title('Distribution of Transaction Amounts by Fraud Status')
plt.xlabel('Transaction Amount')
plt.ylabel('Count')
plt.xlim(0, 2000)  # Limit x-axis for better visualization
plt.show()

In [ ]:
# Compare transaction amounts for fraud vs. non-fraud
plt.figure(figsize=(10, 6))
sns.boxplot(x='is_fraud', y='amt', data=train_data)
plt.title('Transaction Amounts by Fraud Status')
plt.xlabel('Is Fraud (1 = Yes, 0 = No)')
plt.ylabel('Transaction Amount')
plt.ylim(0, 2000)  # Limit y-axis for better visualization
plt.show()

## Categorical Features Analysis

In [ ]:
# Analyze fraud by category
category_fraud = train_data.groupby('category')['is_fraud'].mean().sort_values(ascending=False).reset_index()
category_fraud.columns = ['Category', 'Fraud Rate']

plt.figure(figsize=(12, 8))
sns.barplot(x='Fraud Rate', y='Category', data=category_fraud)
plt.title('Fraud Rate by Transaction Category')
plt.xlabel('Fraud Rate')
plt.ylabel('Category')

# Add percentage labels
for i, rate in enumerate(category_fraud['Fraud Rate']):
    plt.text(rate + 0.001, i, f'{rate:.2%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Top merchants with highest fraud rates (minimum 100 transactions)
merchant_counts = train_data['merchant'].value_counts()
merchants_with_min_trans = merchant_counts[merchant_counts >= 100].index

merchant_fraud = train_data[train_data['merchant'].isin(merchants_with_min_trans)]
merchant_fraud = merchant_fraud.groupby('merchant')['is_fraud'].agg(['mean', 'count'])
merchant_fraud.columns = ['Fraud Rate', 'Transaction Count']
merchant_fraud = merchant_fraud.sort_values('Fraud Rate', ascending=False).head(15).reset_index()

plt.figure(figsize=(14, 8))
sns.barplot(x='Fraud Rate', y='merchant', data=merchant_fraud)
plt.title('Top 15 Merchants with Highest Fraud Rates (Min. 100 Transactions)')
plt.xlabel('Fraud Rate')
plt.ylabel('Merchant')

# Add percentage and count labels
for i, (rate, count) in enumerate(zip(merchant_fraud['Fraud Rate'], merchant_fraud['Transaction Count'])):
    plt.text(rate + 0.001, i, f'{rate:.2%} ({count:,} trans)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Temporal Analysis

In [ ]:
# Convert transaction time to datetime
train_data['trans_date_trans_time'] = pd.to_datetime(train_data['trans_date_trans_time'])

# Extract hour of day
train_data['hour'] = train_data['trans_date_trans_time'].dt.hour

# Analyze fraud by hour of day
hour_fraud = train_data.groupby('hour')['is_fraud'].agg(['mean', 'count']).reset_index()
hour_fraud.columns = ['Hour', 'Fraud Rate', 'Transaction Count']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Plot fraud rate by hour
sns.lineplot(x='Hour', y='Fraud Rate', data=hour_fraud, marker='o', ax=ax1)
ax1.set_title('Fraud Rate by Hour of Day')
ax1.set_ylabel('Fraud Rate')
ax1.grid(True)

# Add percentage labels
for i, rate in enumerate(hour_fraud['Fraud Rate']):
    ax1.text(i, rate + 0.001, f'{rate:.2%}', ha='center', fontsize=9)

# Plot transaction count by hour
sns.barplot(x='Hour', y='Transaction Count', data=hour_fraud, ax=ax2)
ax2.set_title('Transaction Count by Hour of Day')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Transaction Count')

plt.tight_layout()
plt.show()

In [ ]:
# Extract day of week
train_data['day_of_week'] = train_data['trans_date_trans_time'].dt.dayofweek
train_data['day_name'] = train_data['trans_date_trans_time'].dt.day_name()

# Analyze fraud by day of week
day_fraud = train_data.groupby(['day_of_week', 'day_name'])['is_fraud'].agg(['mean', 'count']).reset_index()
day_fraud.columns = ['Day of Week', 'Day Name', 'Fraud Rate', 'Transaction Count']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Plot fraud rate by day of week
sns.barplot(x='Day Name', y='Fraud Rate', data=day_fraud, ax=ax1)
ax1.set_title('Fraud Rate by Day of Week')
ax1.set_ylabel('Fraud Rate')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# Add percentage labels
for i, rate in enumerate(day_fraud['Fraud Rate']):
    ax1.text(i, rate + 0.001, f'{rate:.2%}', ha='center', fontsize=10)

# Plot transaction count by day of week
sns.barplot(x='Day Name', y='Transaction Count', data=day_fraud, ax=ax2)
ax2.set_title('Transaction Count by Day of Week')
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Transaction Count')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## Geographic Analysis

In [ ]:
# Calculate distance between cardholder and merchant
from geopy.distance import geodesic

def calculate_distance(row):
    try:
        cardholder_coords = (row['lat'], row['long'])
        merchant_coords = (row['merch_lat'], row['merch_long'])
        return geodesic(cardholder_coords, merchant_coords).kilometers
    except:
        return np.nan

# Calculate distance for a sample of the data (for performance)
sample_data = train_data.sample(n=10000, random_state=42)
sample_data['distance_km'] = sample_data.apply(calculate_distance, axis=1)

# Analyze distance vs. fraud
plt.figure(figsize=(12, 6))
sns.boxplot(x='is_fraud', y='distance_km', data=sample_data)
plt.title('Distance Between Cardholder and Merchant by Fraud Status')
plt.xlabel('Is Fraud (1 = Yes, 0 = No)')
plt.ylabel('Distance (km)')
plt.ylim(0, 5000)  # Limit y-axis for better visualization
plt.show()

In [ ]:
# Analyze fraud by state
state_fraud = train_data.groupby('state')['is_fraud'].agg(['mean', 'count']).reset_index()
state_fraud.columns = ['State', 'Fraud Rate', 'Transaction Count']
state_fraud = state_fraud[state_fraud['Transaction Count'] >= 1000].sort_values('Fraud Rate', ascending=False)

plt.figure(figsize=(14, 8))
sns.barplot(x='Fraud Rate', y='State', data=state_fraud.head(15))
plt.title('Top 15 States with Highest Fraud Rates (Min. 1000 Transactions)')
plt.xlabel('Fraud Rate')
plt.ylabel('State')

# Add percentage and count labels
for i, (rate, count) in enumerate(zip(state_fraud.head(15)['Fraud Rate'], state_fraud.head(15)['Transaction Count'])):
    plt.text(rate + 0.001, i, f'{rate:.2%} ({count:,} trans)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Select numerical columns for correlation analysis
numerical_cols = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'is_fraud']

# Calculate correlation matrix
correlation_matrix = train_data[numerical_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## Age Analysis

In [ ]:
# Convert DOB to datetime
train_data['dob'] = pd.to_datetime(train_data['dob'])

# Calculate age at the time of transaction
train_data['age'] = train_data.apply(lambda row: (row['trans_date_trans_time'].year - row['dob'].year) - 
                                   ((row['trans_date_trans_time'].month, row['trans_date_trans_time'].day) < 
                                    (row['dob'].month, row['dob'].day)), axis=1)

# Create age groups
bins = [0, 18, 25, 35, 45, 55, 65, 100]
labels = ['<18', '18-25', '26-35', '36-45', '46-55', '56-65', '65+']
train_data['age_group'] = pd.cut(train_data['age'], bins=bins, labels=labels)

# Analyze fraud by age group
age_fraud = train_data.groupby('age_group')['is_fraud'].agg(['mean', 'count']).reset_index()
age_fraud.columns = ['Age Group', 'Fraud Rate', 'Transaction Count']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Plot fraud rate by age group
sns.barplot(x='Age Group', y='Fraud Rate', data=age_fraud, ax=ax1)
ax1.set_title('Fraud Rate by Age Group')
ax1.set_ylabel('Fraud Rate')

# Add percentage labels
for i, rate in enumerate(age_fraud['Fraud Rate']):
    ax1.text(i, rate + 0.001, f'{rate:.2%}', ha='center', fontsize=10)

# Plot transaction count by age group
sns.barplot(x='Age Group', y='Transaction Count', data=age_fraud, ax=ax2)
ax2.set_title('Transaction Count by Age Group')
ax2.set_xlabel('Age Group')
ax2.set_ylabel('Transaction Count')

plt.tight_layout()
plt.show()

## Key Findings and Insights

Based on the exploratory data analysis, here are the key findings and insights:

1. **Class Imbalance**: The dataset is highly imbalanced, with fraudulent transactions representing only a small percentage of the total transactions.

2. **Transaction Amount**: Fraudulent transactions tend to have different amount patterns compared to legitimate transactions. There appears to be a higher fraud rate for certain transaction amount ranges.

3. **Merchant Categories**: Some merchant categories have significantly higher fraud rates than others. This could be a strong predictor for fraud detection.

4. **Temporal Patterns**: Fraud rates vary by hour of day and day of week, suggesting that time-based features could be valuable for fraud detection.

5. **Geographic Factors**: The distance between the cardholder and merchant locations appears to be a potential indicator of fraud. Certain states also have higher fraud rates.

6. **Age Groups**: Fraud rates vary across different age groups, indicating that age could be a useful feature for fraud detection.

These insights will guide our feature engineering process to create effective predictive features for the fraud detection model.

## Next Steps

Based on the EDA findings, the next steps for the project are:

1. **Feature Engineering**:
   - Create time-based features (hour, day, weekday, month)
   - Calculate distance between cardholder and merchant
   - Derive age from date of birth
   - Create features for transaction amount relative to category average
   - Encode categorical variables

2. **Model Selection and Training**:
   - Address class imbalance using techniques like SMOTE
   - Train multiple classification models
   - Optimize hyperparameters
   - Evaluate models using appropriate metrics (precision, recall, F1-score)

3. **Model Deployment**:
   - Implement the API for real-time fraud prediction
   - Create a web UI for demonstration

The next notebook will focus on feature engineering based on these insights.